### De novo aptamer design using AptaDiff
Step-by-step guide for generating new aptamer sequences from raw SELEX reads.

### Overview

AptaDiff generates new aptamers from the reads of a SELEX experiment. It learns directly from the sequences that survived selection.

Training happens in three stages: 
- a pretrained variational autoencoder (VAE) maps each read to a point in a small latent space. 
- A diffusion model learns to generate a read back from its own latent point. 
- A Gaussian mixture is then fitted to those points, and new sequences are generated from it.

The classes used in this notebook:

* `MoleculeLoader`: lazy container for sequence data, cells may hold sequences or file paths, materialized with `to_dataframe()`.
* `PrimerTrimmer`: strips the constant primer regions of a SELEX read and keeps the random region.
* `SequenceIndexEncoder`: turns the trimmed reads into integer indices, the input of the VAE.
* `RaptGenModel`: trains the RaptGen VAE and supplies the latent coordinates.
* `AptaDiffGenerator`: trains the diffusion model and generates a sequence from a latent point.
* `AptaDiff`: the end-to-end estimator that ties these together, with `fit` and `sample`.

The data is `datasetD.fastq`, the SELEX library published with the AptaDiff paper.


In [ ]:
from pyaptamer.aptadiff import AptaDiff, AptaDiffGenerator
from pyaptamer.data import MoleculeLoader
from pyaptamer.raptgen import RaptGenModel
from pyaptamer.transformations.encode import SequenceIndexEncoder
from pyaptamer.transformations.transform import PrimerTrimmer

### Settings

This is an example run and can be done in few minutes on CPU


In [ ]:
# Data
DATA_PATH = "data_selex/datasetD_wang2024.fastq"
READ_LENGTH = 100
N_READS = 20000  # subset of reads used for training, the full file holds 121,185

# SELEX library design
START_PRIMER = "TAATACGACTCACTATAGGGAGAACTTCGACCAGAAG"
END_PRIMER = "TATGTGCGCATACATGGATCCTC"
VARIABLE_LENGTH = 40

# VAE
VAE_EPOCHS = 3
BETA_THRESHOLD = 2  # epochs over which the KL weight is ramped to 1.0


# Diffusion Model
DIM = 128
DEPTH = 4
NUM_TIMESTEPS = 100
MAX_EPOCHS = 8

# Latent space and clustering
EMBED_SIZE = 2
N_CLUSTERS = 8
GMM_RESTARTS = 100

RANDOM_STATE = 0

### Step 1: Load the SELEX reads

Every read follows the same library design: a constant 37-nt start primer, a 40-nt random region, then a constant 23-nt end primer. Reads that were truncated during sequencing are shorter than 100 nt, so we keep only the full-length ones.


In [ ]:
loader = MoleculeLoader(data={"sequence": [DATA_PATH]}, tiling="samples")
reads = loader.to_dataframe()

print(f"{len(reads)} reads in the file")
reads.head()

In [ ]:
reads["sequence"].str.len().value_counts().head(4)

In [ ]:
X = reads[reads["sequence"].str.len() == READ_LENGTH].head(N_READS)

print(f"{len(X)} full-length reads selected")
X.head()

### Step 2: Strip the primers

Only the random region is a candidate aptamer. `PrimerTrimmer` removes the two constant regions and drops any read that does not match the design, for example a read with a sequencing error inside a primer.


In [ ]:
trimmer = PrimerTrimmer(
    start_primer=START_PRIMER,
    end_primer=END_PRIMER,
    variable_length=VARIABLE_LENGTH,
)

trimmed = trimmer.fit_transform(X)
trimmed.head()

### Step 3: Build the components

AptaDiff takes three parts: a VAE, a trimmer and a diffusion generator. Each is built on its own, so each can be configured or swapped without touching the others.

The VAE's encoder compresses each 40-nt read into `EMBED_SIZE` numbers. Reads with similar sequences will lie close to each other in latent space.

The diffusion model will learn to do the reverse, turning this point in latent space back into a sequence.

Using `fast=True` picks `CNNPHMMVAEFast`, whose profile-HMM loss runs about ten times faster per batch than the standard decoder. `patience` handles early stopping and also the number of epochs over which the KL weight is ramped up to one, as in the original RaptGen code.

In [ ]:
X_idx = SequenceIndexEncoder().fit_transform(trimmed)

print(f"{len(X_idx)} reads encoded as {X_idx.shape[1]}-nt index vectors")

In [ ]:
vae = RaptGenModel(
    motif_len=VARIABLE_LENGTH,
    embed_size=EMBED_SIZE,
    fast=True,
    max_epochs=VAE_EPOCHS,
    patience=BETA_THRESHOLD,
    random_state=RANDOM_STATE,
).fit(X_idx)

In [ ]:
generator = AptaDiffGenerator(
    num_timesteps=NUM_TIMESTEPS,
    dim=DIM,
    depth=DEPTH,
    max_epochs=MAX_EPOCHS,
    random_state=RANDOM_STATE,
    # verbose=1, # shows train_loss/val_loss per epoch in a Lightning progress bar.
)

### Step 4: Fit

`fit` trims and encodes the reads, reads their latent coordinates out of the VAE, trains the diffusion model, and fits the Gaussian mixture. With the settings above this takes roughly three to four minutes on a CPU.

In [ ]:
aptadiff = AptaDiff(
    vae=vae,
    trimmer=trimmer,
    generator=generator,
    n_clusters=N_CLUSTERS,
    gmm_restarts=GMM_RESTARTS,
    random_state=RANDOM_STATE,
)

aptadiff.fit(X)

In [ ]:
print("sequences kept: ", aptadiff.sequences_.shape)
print("latent coordinates: ", aptadiff.latents_.shape)
print("sequence length: ", aptadiff.seq_len_)
print("cluster centers: ", aptadiff.gmm_centers_.shape)

### Step 5: Generate new sequences

`sample` draws latent points from the fitted mixture and generates one sequence per point. There is no `predict`. AptaDiff is generative, so new sequences come from `sample`.

In [ ]:
aptadiff.sample(10)

### Step 6: Design from the cluster centers

The mixture components mark the dense regions of the latent space. Generating from the component means, instead of from random draws, is the de novo design procedure used in the original paper. `inverse_transform` accepts any latent points, including ones proposed by an outer optimisation loop such as Bayesian optimisation over wet-lab results.

In [ ]:
aptadiff.inverse_transform(aptadiff.gmm_centers_)

### Step 7: Plot the latent space (optional)

`EMBED_SIZE` is 2, so the latent space can be plotted directly. Each point is one training read, colored by the mixture component it is assigned to. The ellipses are the two-sigma contours of the components, the regions `sample` draws from, and the numbered dots are their centers, which Step 6 generates from. The hollow circles are ten draws from the mixture, the latent points behind ten new sequences.

With a VAE trained for three epochs the structure is coarse. It sharpens with longer VAE training. The plot needs matplotlib, installed with `pip install "pyaptamer[plot]"`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse

latents = aptadiff.latents_
centers = aptadiff.gmm_centers_
component = aptadiff.gmm_.predict(latents)
draws, _ = aptadiff.gmm_.sample(10)
cmap = plt.get_cmap("tab10")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(latents[:, 0], latents[:, 1], c=cmap(component), s=5, alpha=0.3)

for k, cov in enumerate(aptadiff.gmm_.covariances_):
    eigval, eigvec = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(eigvec[1, 1], eigvec[0, 1]))
    width, height = 4 * np.sqrt(eigval[::-1])
    ax.add_patch(
        Ellipse(centers[k], width, height, angle=angle, fill=False, color=cmap(k))
    )
    ax.annotate(str(k), centers[k], xytext=(6, 6), textcoords="offset points")

ax.scatter(centers[:, 0], centers[:, 1], color="black", s=40, label="component center")
ax.scatter(
    draws[:, 0],
    draws[:, 1],
    facecolor="none",
    edgecolor="black",
    s=40,
    label="draw from the mixture",
)
ax.set(
    xlabel="latent 1", ylabel="latent 2", title="Training reads by mixture component"
)
ax.legend()
plt.show()

### Step 8: Score sequences

`score` reports the mean log-likelihood of the sequences in bits per character. Higher is better. It is a quick way to compare runs, for example after training the VAE for more epochs.

In [ ]:
print("bits per character:", aptadiff.score(X.head(500)))
aptadiff.score_samples(X.head(5))